In [1]:
!pip install -q python-telegram-bot nest_asyncio

In [4]:
import os
from getpass import getpass

BOT_TOKEN = getpass("Paste your Telegram Bot Token here: ")

if BOT_TOKEN:
    os.environ["BOT_TOKEN"] = BOT_TOKEN
    print("Bot token saved successfully!")
else:
    print("Token was not entered.")

Paste your Telegram Bot Token here: ··········
Bot token saved successfully!


In [6]:
from telegram import Bot
import os
import asyncio

async def test_connection():
    bot = Bot(token=os.environ["BOT_TOKEN"])

    bot_info = await bot.get_me()

    print("Bot connected successfully!")
    print("Bot Name:", bot_info.first_name)
    print("Bot Username:", bot_info.username)

await test_connection()

Bot connected successfully!
Bot Name: HFAssist
Bot Username: HFAssist_Project_Bot


In [7]:
from telegram import Update
from telegram.ext import (
    Application,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters
)

# Response to /start command
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Welcome to HFAssist! Send me a message."
    )

# Response to normal text messages
async def handle_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_text = update.message.text

    await update.message.reply_text(
        f"You sent: {user_text}"
    )

# Create the application
application = Application.builder().token(BOT_TOKEN).build()

# Register handlers
application.add_handler(CommandHandler("start", start))
application.add_handler(
    MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text)
)

# Start the bot
await application.initialize()
await application.start()
await application.updater.start_polling()

print("HFAssist is now running!")

HFAssist is now running!


In [8]:
!pip install -q "transformers<5" pillow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [9]:
import transformers
import telegram

print("Transformers:", transformers.__version__)
print("Telegram:", telegram.__version__)

Transformers: 4.57.6
Telegram: 22.8


In [10]:
from transformers import pipeline

image_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

print("Image Classification model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu


Image Classification model loaded successfully!


In [12]:
async def handle_image(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text(
        "Image received! Analyzing..."
    )

    photo = update.message.photo[-1]

    file = await context.bot.get_file(photo.file_id)

    image_path = "telegram_image.jpg"

    await file.download_to_drive(image_path)

    result = image_classifier(image_path, top_k=3)

    response = "Image Classification Results:\n\n"

    for item in result:
        response += f"{item['label']}: {item['score']*100:.2f}%\n"

    await update.message.reply_text(response)


application.add_handler(
    MessageHandler(filters.PHOTO, handle_image)
)

print("Image Classification connected to Telegram!")

Image Classification connected to Telegram!


In [13]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr"
)

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("All Text Pipeline models loaded successfully!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


config.json: 0.00B [00:00, ?B/s]

... (20 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


All Text Pipeline models loaded successfully!


In [19]:
async def handle_text(update: Update, context: ContextTypes.DEFAULT_TYPE):

    message = update.message.text
    parts = message.split(maxsplit=1)

    command = parts[0].split("@")[0].lower()
    text = parts[1].strip() if len(parts) > 1 else ""

    if not text:
        await update.message.reply_text(
            "Please provide text after the command."
        )
        return

    await update.message.reply_text("Processing your request...")

    try:
        if command == "/sentiment":
            result = sentiment_model(text)[0]
            response = (
                f"Sentiment: {result['label']}\n"
                f"Confidence: {result['score']*100:.2f}%"
            )

        elif command == "/summarize":
            result = summarizer(
                text, max_new_tokens=100, do_sample=False
            )[0]
            response = result["summary_text"]

        elif command == "/translate":
            result = translator(text)[0]
            response = result["translation_text"]

        elif command == "/generate":
            result = generator(
                text,
                max_new_tokens=60,
                num_return_sequences=1,
                pad_token_id=generator.tokenizer.eos_token_id
            )[0]
            response = result["generated_text"]

        else:
            response = "Unknown command."

        await update.message.reply_text(response)

    except Exception as e:
        await update.message.reply_text(f"Error: {str(e)}")


for group, handlers in list(application.handlers.items()):
    for handler in list(handlers):
        if (
            isinstance(handler, MessageHandler)
            and handler.callback.__name__ == "handle_text"
        ):
            application.remove_handler(handler, group=group)


application.add_handler(
    MessageHandler(filters.COMMAND, handle_text)
)

print("Updated Text Handler Registered Successfully!")


Updated Text Handler Registered Successfully!


In [20]:
!pip install -q librosa soundfile

In [21]:
from transformers import pipeline

audio_transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny"
)

print("Audio model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Audio model loaded successfully!


In [22]:
async def handle_audio(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text(
        "Audio received! Transcribing..."
    )

    audio_file = update.message.voice or update.message.audio

    file = await context.bot.get_file(audio_file.file_id)

    audio_path = "telegram_audio.ogg"

    await file.download_to_drive(audio_path)

    try:
        result = audio_transcriber(audio_path)

        transcription = result["text"]

        await update.message.reply_text(
            f"Transcription:\n\n{transcription}"
        )

    except Exception as e:
        await update.message.reply_text(
            f"Audio processing error: {str(e)}"
        )


application.add_handler(
    MessageHandler(
        filters.VOICE | filters.AUDIO,
        handle_audio
    )
)

print("Audio Pipeline connected to Telegram!")

Audio Pipeline connected to Telegram!


In [1]:
from transformers import pipeline

video_classifier = pipeline(
    "video-classification",
    model="MCG-NJU/videomae-base-finetuned-kinetics"
)

print("Video Classification Model Loaded!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


Video Classification Model Loaded!


In [2]:
import cv2
print("OpenCV Ready!")

OpenCV Ready!


In [24]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 59.3 MB/s  0:00:00


/usr/lib/python3.13/pathlib/_local.py:274: RuntimeWarning: coroutine 'test_connection' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [5]:
async def handle_video(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text(
        "Video received! Analyzing..."
    )

    video = update.message.video

    file = await context.bot.get_file(video.file_id)

    video_path = "telegram_video.mp4"

    await file.download_to_drive(video_path)

    try:
        result = video_classifier(video_path, top_k=3)

        response = "Video Classification Results:\n\n"

        for item in result:
            response += (
                f"{item['label']}: "
                f"{item['score']*100:.2f}%\n"
            )

        await update.message.reply_text(response)

    except Exception as e:
        await update.message.reply_text(
            f"Video processing error: {str(e)}"
        )


application.add_handler(
    MessageHandler(filters.VIDEO, handle_video)
)

print("Video Pipeline connected to Telegram!")

NameError: name 'application' is not defined

In [4]:
from telegram import Update
from telegram.ext import (
    Application,
    MessageHandler,
    CommandHandler,
    ContextTypes,
    filters
)

print("Telegram imports ready!")

Telegram imports ready!


In [6]:
application = Application.builder().token(BOT_TOKEN).build()

application.add_handler(
    CommandHandler("start", start)
)

application.add_handler(
    MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text)
)

application.add_handler(
    MessageHandler(filters.PHOTO, handle_image)
)

application.add_handler(
    MessageHandler(filters.VOICE | filters.AUDIO, handle_audio)
)

print("Telegram Application Created Successfully!")

NameError: name 'BOT_TOKEN' is not defined

In [7]:
import os
from getpass import getpass

BOT_TOKEN = getpass("Enter your Telegram Bot Token: ")

os.environ["BOT_TOKEN"] = BOT_TOKEN

print("Bot Token Loaded Successfully!")

Enter your Telegram Bot Token: ··········
Bot Token Loaded Successfully!


In [8]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

translator = pipeline(
    "translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr"
)

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

image_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

audio_transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny"
)

print("All Previous Models Restored Successfully!")

Device set to use cpu
Device set to use cpu
/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu
Device set to use cpu
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu
Device set to use cpu


All Previous Models Restored Successfully!


In [9]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Welcome to HFAssist! 🤖\n\n"
        "Available Features:\n"
        "1. Text Analysis — /sentiment, /summarize, /translate, /generate\n"
        "2. Image Classification — Send an image\n"
        "3. Audio Transcription — Send a voice message\n"
        "4. Video Classification — Send a video"
    )

print("Start function ready!")

Start function ready!


In [10]:
async def handle_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    message = update.message.text
    parts = message.split(maxsplit=1)

    command = parts[0].split("@")[0].lower()
    text = parts[1].strip() if len(parts) > 1 else ""

    if not text:
        await update.message.reply_text("Please provide some text after the command.")
        return

    await update.message.reply_text("Processing your request...")

    try:
        if command == "/sentiment":
            result = sentiment_model(text)[0]
            response = (
                f"Sentiment: {result['label']}\n"
                f"Confidence: {result['score']*100:.2f}%"
            )

        elif command == "/summarize":
            result = summarizer(text, max_new_tokens=100, do_sample=False)[0]
            response = result["summary_text"]

        elif command == "/translate":
            result = translator(text)[0]
            response = result["translation_text"]

        elif command == "/generate":
            result = generator(
                text,
                max_new_tokens=60,
                num_return_sequences=1,
                pad_token_id=generator.tokenizer.eos_token_id
            )[0]
            response = result["generated_text"]

        else:
            response = "Unknown command."

        await update.message.reply_text(response)

    except Exception as e:
        await update.message.reply_text(f"Error: {str(e)}")

print("Text handler ready!")

Text handler ready!


In [11]:
async def handle_image(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text("Image received! Analyzing...")

    photo = update.message.photo[-1]
    file = await context.bot.get_file(photo.file_id)

    image_path = "telegram_image.jpg"
    await file.download_to_drive(image_path)

    try:
        result = image_classifier(image_path, top_k=3)

        response = "Image Classification Results:\n\n"

        for item in result:
            response += f"{item['label']}: {item['score']*100:.2f}%\n"

        await update.message.reply_text(response)

    except Exception as e:
        await update.message.reply_text(f"Image processing error: {str(e)}")

print("Image handler ready!")

Image handler ready!


In [12]:
async def handle_audio(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text("Audio received! Transcribing...")

    audio_file = update.message.voice or update.message.audio

    file = await context.bot.get_file(audio_file.file_id)

    audio_path = "telegram_audio.ogg"
    await file.download_to_drive(audio_path)

    try:
        result = audio_transcriber(audio_path)
        transcription = result["text"]

        await update.message.reply_text(
            f"Transcription:\n\n{transcription}"
        )

    except Exception as e:
        await update.message.reply_text(
            f"Audio processing error: {str(e)}"
        )

print("Audio handler ready!")

Audio handler ready!


In [13]:
async def handle_video(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text("Video received! Analyzing...")

    video = update.message.video
    file = await context.bot.get_file(video.file_id)

    video_path = "telegram_video.mp4"
    await file.download_to_drive(video_path)

    try:
        result = video_classifier(video_path, top_k=3)

        response = "Video Classification Results:\n\n"

        for item in result:
            response += f"{item['label']}: {item['score']*100:.2f}%\n"

        await update.message.reply_text(response)

    except Exception as e:
        await update.message.reply_text(
            f"Video processing error: {str(e)}"
        )

print("Video handler ready!")

Video handler ready!


In [14]:
application = Application.builder().token(BOT_TOKEN).build()

print("HFAssist application created!")

HFAssist application created!


In [15]:
application.add_handler(CommandHandler("start", start))

application.add_handler(
    MessageHandler(filters.COMMAND, handle_text)
)

application.add_handler(
    MessageHandler(filters.PHOTO, handle_image)
)

application.add_handler(
    MessageHandler(filters.VOICE | filters.AUDIO, handle_audio)
)

application.add_handler(
    MessageHandler(filters.VIDEO, handle_video)
)

print("All 5 handlers registered successfully!")

All 5 handlers registered successfully!


In [16]:
await application.initialize()
await application.start()
await application.updater.start_polling()

print("HFAssist Bot is Running!")

HFAssist Bot is Running!


In [17]:
print("Video handler registered:", handle_video)
print("Video filter:", filters.VIDEO)
print("Bot running:", application.running)

Video handler registered: <function handle_video at 0x7fa43627c680>
Video filter: filters.VIDEO
Bot running: True


In [18]:
async def debug_video(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print("MESSAGE RECEIVED!")
    print("Video:", update.message.video)
    print("Document:", update.message.document)
    print("Animation:", update.message.animation)

application.add_handler(
    MessageHandler(
        filters.ALL,
        debug_video
    ),
    group=1
)

print("Debug handler added!")

Debug handler added!
